# IMPORT

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import pydeck as pdk
import folium


# DEFINE FOLDERS

In [2]:
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
PLOT_FOLDER = Path("plots")

folders = [DATASET_INPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [3]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_data.csv")
df.head(5)

,timestamp,station_code,latitude,longitude,capacity,total_bikes_now,total_bikes_taken,temperature,wind_speed,wind_direction,weather_code,is_day,hour,dayofweek,is_weekend,sin_hour,cos_hour
0,2025-11-04 04:00:00,1001,48.857059,2.341798,17.0,2.0,15.0,10.101562,9.500000,217.0,3.000000,0.000000,4,1,0,0.866025,0.500000
1,2025-11-04 04:15:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.094435,9.283236,inf,14.660302,0.365327,4,1,0,0.866025,0.500000
2,2025-11-04 04:30:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.000000,8.000000,207.0,3.000000,0.000000,4,1,0,0.866025,0.500000
3,2025-11-04 04:45:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.101562,8.203125,203.0,3.000000,0.000000,4,1,0,0.866025,0.500000
4,2025-11-04 05:00:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.094435,9.283236,inf,14.660302,0.365327,5,1,0,0.965926,0.258819


# 

In [4]:
def save_unique_stations_map_html(
    df,
    out_html="idf_stations.html",
    station_col="station_code",
    lat_col="latitude",
    lon_col="longitude",
    radius=4,
    color="red",
    fill_opacity=0.8,
    zoom_start=10,
):
    # 1) Keep only the columns we need + drop missing coords
    d = df[[station_col, lat_col, lon_col]].dropna(subset=[lat_col, lon_col])

    # 2) Keep ONE row per station (huge speedup vs 4M rows)
    #    If station_code is unique identifier, this is the best:
    d = d.drop_duplicates(subset=[station_col], keep="first")

    # 3) Ensure numeric
    d[lat_col] = d[lat_col].astype(float)
    d[lon_col] = d[lon_col].astype(float)

    # 4) Create blank map (only your points, no basemap clutter)
    center_lat = float(d[lat_col].mean())
    center_lon = float(d[lon_col].mean())
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles=None)

    # 5) Plot ~1502 points (fast)
    for code, la, lo in zip(d[station_col], d[lat_col], d[lon_col]):
        folium.CircleMarker(
            location=[la, lo],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=fill_opacity,
            tooltip=str(code),
        ).add_to(m)

    m.save(out_html)
    return out_html, d.shape[0]  # returns file path + how many unique stations mapped

# Example:
html_path, n = save_unique_stations_map_html(df, radius=5)
print("Saved:", html_path, "| stations mapped:", n)

Saved: idf_stations.html | stations mapped: 1438


In [5]:
def save_unique_stations_map_html_minimal(
    df,
    out_html="idf_stations_minimal_2.html",
    station_col="station_code",
    lat_col="latitude",
    lon_col="longitude",
    radius=4,
    color="red",
    fill_opacity=0.8,
    zoom_start=10,
    tiles="CartoDB positronNoLabels",  # minimal; try "CartoDB positron" too
):
    # Keep only needed cols, drop missing coords
    d = df[[station_col, lat_col, lon_col]].dropna(subset=[lat_col, lon_col])

    # Keep one row per station (huge speedup vs millions of rows)
    d = d.drop_duplicates(subset=[station_col], keep="first")

    # Ensure numeric
    d[lat_col] = d[lat_col].astype(float)
    d[lon_col] = d[lon_col].astype(float)

    # Center map
    center_lat = float(d[lat_col].mean())
    center_lon = float(d[lon_col].mean())

    # Minimal basemap
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles=tiles)

    # Plot points
    for code, la, lo in zip(d[station_col], d[lat_col], d[lon_col]):
        folium.CircleMarker(
            location=[la, lo],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=fill_opacity,
            tooltip=str(code),
        ).add_to(m)

    m.save(out_html)
    return out_html, len(d)

html_path, n = save_unique_stations_map_html_minimal(df, radius=0.1)
print("Saved:", html_path, "| stations mapped:", n)

Saved: idf_stations_minimal_2.html | stations mapped: 1438


# 

In [6]:
def stations_map_blank(df, lat_col="latitude", lon_col="longitude",
                       tooltip_col="station_code",
                       radius=3, color="red", fill_color="red", fill_opacity=0.8):
    # Center map
    center_lat = df[lat_col].astype(float).mean()
    center_lon = df[lon_col].astype(float).mean()

    # tiles=None => no map features (no roads, hospitals, etc.)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)

    for _, r in df.iterrows():
        folium.CircleMarker(
            location=[float(r[lat_col]), float(r[lon_col])],
            radius=radius,                 # change this anytime
            color=color,
            fill=True,
            fill_color=fill_color,
            fill_opacity=fill_opacity,
            tooltip=str(r[tooltip_col]) if tooltip_col in df.columns else None
        ).add_to(m)

    return m

In [ ]:
m = stations_map_blank(df, radius=4)   # try 2, 4, 6, ...
m

# 

In [ ]:
layer = pdk.Layer(
    "ScatterplotLayer",
    data=df,
    get_position='[longitude, latitude]',
    get_radius=25,         # meters-ish visually; tweak
    pickable=True
)

view = pdk.ViewState(
    latitude=df["latitude"].mean(),
    longitude=df["longitude"].mean(),
    zoom=9
)

deck = pdk.Deck(layers=[layer], initial_view_state=view, tooltip={"text": "{station_code}"})
deck.to_html("idf_stations.html", open_browser=True)

# 

In [ ]:
fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    hover_name="station_code",          # optional
    zoom=9,
    height=700
)

fig.update_layout(map_style="open-street-map")
fig.show()

# 

In [ ]:
fig = px.density_map(
    df, lat="latitude", lon="longitude",
    zoom=9, radius=6, height=700
)
fig.update_layout(map_style="open-street-map")
fig.show()